<a href="https://colab.research.google.com/github/Brainspark1/Robot-Voice-Project/blob/main/Robot_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install librosa sounddevice numpy matplotlib tensorflow

In [ ]:
!pip install pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
!unzip commands_zip.zip -d commands_data

Archive:  commands_zip.zip
replace commands_data/LICENSE? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import os

DATASET_PATH = "commands_data"

COMMANDS = ["go", "left", "right", "stop"]

rows = []

for folder in os.listdir(DATASET_PATH):
    folder_path = os.path.join(DATASET_PATH, folder)

    if not os.path.isdir(folder_path):
        continue

    if folder == "_background_noise_":
        continue

    for file in os.listdir(folder_path):
        if not file.endswith(".wav"):
            continue

        file_path = os.path.join(folder_path, file)

        label = folder if folder in COMMANDS else "unknown"

        rows.append({
            "file_path": file_path,
            "label": label
        })

df = pd.DataFrame(rows)
df.head()

print("Folders found:", os.listdir(DATASET_PATH))
print(df["label"].value_counts())

Folders found: ['yes', 'one', 'right', 'validation_list.txt', 'dog', 'LICENSE', 'sheila', 'happy', 'zero', 'six', '_background_noise_', 'nine', 'bed', 'down', 'off', 'README.md', 'marvin', 'seven', 'five', 'go', 'cat', 'no', 'on', 'testing_list.txt', 'two', 'bird', 'wow', 'eight', 'left', 'four', 'tree', 'three', 'stop', 'up', 'house']
label
unknown    55249
stop        2380
go          2372
right       2367
left        2353
Name: count, dtype: int64


In [ ]:
df = df[df["label"] != "unknown"].reset_index(drop=True)

df['label'].value_counts()

,count
label,
stop,2380
go,2372
right,2367
left,2353


In [ ]:
import librosa

def extract_mfcc(file_path, max_len=40):
    audio, sr = librosa.load(file_path, sr=16000)

    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)

    if mfcc.shape[1] < max_len:
        pad_width = max_len - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0,0),(0,pad_width)))
    else:
        mfcc = mfcc[:, :max_len]

    return mfcc

In [ ]:
X = np.array([extract_mfcc(f) for f in df["file_path"]])
y = df["label"].values

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

print(le.classes_)

['go' 'left' 'right' 'stop']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(40,40,1)),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(4, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 11s 41ms/step - accuracy: 0.6583 - loss: 1.1166 - val_accuracy: 0.8739 - val_loss: 0.3421
Epoch 2/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - accuracy: 0.8981 - loss: 0.2926 - val_accuracy: 0.9404 - val_loss: 0.1770
Epoch 3/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - accuracy: 0.9299 - loss: 0.1953 - val_accuracy: 0.9388 - val_loss: 0.1690
Epoch 4/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - accuracy: 0.9526 - loss: 0.1315 - val_accuracy: 0.9488 - val_loss: 0.1318
Epoch 5/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - accuracy: 0.9642 - loss: 0.0998 - val_accuracy: 0.9562 - val_loss: 0.1304
Epoch 6/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 9s 39ms/step - accuracy: 0.9687 - loss: 0.0814 - val_accuracy: 0.9483 - val_loss: 0.1606
Epoch 7/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - accuracy: 0.9722 - loss: 0.0791 - val_accuracy: 0.9530 - val_loss: 0.1303
Epoch 8/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.9778 - loss: 0.0581 - val_a

In [ ]:
pred = model.predict(X_val[0:1])

index = np.argmax(pred)
label = le.inverse_transform([index])

print(label)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
['right']


In [ ]:
pred = model.predict(X_val[0:10])

for i in range(10):
    index = np.argmax(pred[i])
    label = le.inverse_transform([index])
    print("Pred:", label, "True:", le.inverse_transform([y_val[i]]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
Pred: ['right'] True: ['right']
Pred: ['left'] True: ['go']
Pred: ['stop'] True: ['stop']
Pred: ['go'] True: ['go']
Pred: ['right'] True: ['right']
Pred: ['right'] True: ['right']
Pred: ['right'] True: ['right']
Pred: ['right'] True: ['right']
Pred: ['stop'] True: ['stop']
Pred: ['right'] True: ['right']


In [ ]:
loss, acc = model.evaluate(X_val, y_val)
print("Validation accuracy:", acc)

60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9662 - loss: 0.1212
Validation accuracy: 0.9662269353866577


In [ ]:
model.save("voice_robot_model.h5")